In [22]:
import pandas as pd
from lightfm import LightFM
from lightfm.data import Dataset
from sklearn.model_selection import train_test_split
from lightfm.evaluation import precision_at_k
import numpy as np

In [4]:
eng_df = pd.read_csv("english_interactions.csv")

In [36]:
eng_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5107639 entries, 0 to 5107638
Data columns (total 3 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   user_id     object
 1   article_id  object
 2   clicked     int64 
dtypes: int64(1), object(2)
memory usage: 116.9+ MB


In [6]:
print(eng_df)

        user_id article_id  clicked
0        U13740     N55189        1
1        U13740     N42782        1
2        U13740     N34694        1
3        U13740     N45794        1
4        U13740     N18445        1
...         ...        ...      ...
5107634  U44625     N43083        1
5107635  U44625      N9288        1
5107636  U44625     N37863        1
5107637  U64800     N22997        1
5107638  U64800     N48742        1

[5107639 rows x 3 columns]


In [8]:
train_beh = pd.read_csv(
    'MINDsmall_train/behaviors.tsv',
    sep='\t', header=None,
    names=['impr_id','user_id','ts','history','impr_list'],
    dtype=str
)

In [2]:
print(train_beh)

NameError: name 'train_beh' is not defined

In [10]:
train_beh['history'] = train_beh['history'].fillna('').str.split()
train_rows = [
    (uid, aid)
    for uid, hist in zip(train_beh['user_id'], train_beh['history'])
    for aid in hist
]
train_df = pd.DataFrame(train_rows, columns=['user_id','article_id'])
train_df['clicked'] = 1

In [12]:
val_beh = pd.read_csv(
    'MINDsmall_dev/behaviors.tsv',
    sep='\t', header=None,
    names=['impr_id','user_id','ts','history','impr_list'],
    dtype=str
)
val_beh['history'] = val_beh['history'].fillna('').str.split()
val_rows = [
    (uid, aid)
    for uid, hist in zip(val_beh['user_id'], val_beh['history'])
    for aid in hist
]
val_df = pd.DataFrame(val_rows, columns=['user_id','article_id'])
val_df['clicked'] = 1


In [24]:
# 3) Build Dataset over all users/items in train+val
all_users = pd.concat([train_df['user_id'], val_df['user_id']]).unique()
train_items = train_df['article_id'].unique()
val_items   = val_df  ['article_id'].unique()
all_items   = np.union1d(train_items, val_items)

dataset = Dataset()
dataset.fit(
    users = np.union1d(train_df['user_id'].unique(), val_df['user_id'].unique()),
    items = all_items
)
dataset = Dataset()
dataset.fit(users=all_users, items=all_items)

In [26]:
# 4) Build sparse interaction matrices
train_interactions, _ = dataset.build_interactions(
    train_df[['user_id','article_id']].itertuples(index=False, name=None)
)
val_interactions, _ = dataset.build_interactions(
    val_df[['user_id','article_id']].itertuples(index=False, name=None)
)

using the above train and val interactions caused an error in the model training and evaluation as they had common interaction pairs overlapping. this was a problem since if you count an interaction that the model trained on in the clicks, the model might give unusually high precision. to avoid this, i am filtering the datasets. 

In [31]:
# Build a set of train pairs
train_pairs = set(zip(train_df['user_id'], train_df['article_id']))

# Filter val_df
val_df_filtered = val_df[
    ~val_df.apply(lambda r: (r['user_id'], r['article_id']) in train_pairs, axis=1)
].reset_index(drop=True)

# Now rebuild the interactions matrix
val_interactions, _ = dataset.build_interactions(
    val_df_filtered[['user_id','article_id']].itertuples(index=False, name=None)
)

In [28]:
model = LightFM(no_components=30, loss="warp")
model.fit(train_interactions, epochs=10, num_threads=4)

In [33]:
train_prec = precision_at_k(model, train_interactions, k=10).mean()
val_prec   = precision_at_k(model, val_interactions, train_interactions=train_interactions, k=10).mean()
print(f"Precision@10 → train: {train_prec:.4f}, validation: {val_prec:.4f}")

Precision@10 → train: 0.0970, validation: 0.0796


In [38]:
print(f"Interaction matrices built:")
print(f"   Train: {train_interactions.shape[0]} users × {train_interactions.shape[1]} items")
print(f"   Val:   {val_interactions.shape[0]} users × {val_interactions.shape[1]} items")

Interaction matrices built:
   Train: 91935 users × 44908 items
   Val:   91935 users × 44908 items
